In [2]:
import pandas as pd
import re
from typing import Dict, Tuple

In [3]:
DATASET_RULES = {
    'production': {
        'patterns': [
            r'batch_production_\d+.*',
            r'\d+_myekyc_.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
    'datacollection': {
        'patterns': [
            r'batch_datacollection_\d+.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
    'issue': {
        'patterns': [
            r'batch_issue_\d+.*',
        ],
        'weights': {'apcer': -0.0769, 'bpcer': 0},
    },
    'test_plan': {
        'patterns': [
            r'.*test_plan_subject.*',
            r'grayscale_print.*',
            r'colorprint.*',
            r'replay_.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
}

# Special cases with exact match
SPECIAL_CASES = {
    'batch_issue_set/index_annotation_mykadfront_background.csv': {
        'weights': {'apcer': 0, 'bpcer': -0.0769},
    },
    'batch_issue_20240412/partner_both_general-index_annotation_mykadfront': {
        'weights': {'apcer': -0.0769, 'bpcer': -0.0769},
    }
}

In [4]:
def classify_dataset(dataset_name: str) -> Tuple[str, Dict[str, float]]:
    """Classify dataset and return (category, weights)."""
    # Check special cases first
    if dataset_name in SPECIAL_CASES:
        return 'special', SPECIAL_CASES[dataset_name]['weights']

    # Check against pattern rules
    for category, rules in DATASET_RULES.items():
        for pattern in rules['patterns']:
            if re.match(pattern, dataset_name, re.IGNORECASE):
                return category, rules['weights']

    # Default
    return 'unknown', {'apcer': 1.0, 'bpcer': 1.0}

In [5]:
df = pd.read_csv('metrics_by_epoch.csv')
df.columns

Index(['epoch', 'TP', 'FP', 'TN', 'FN', 'apcer', 'bpcer', 'acer', 'accuracy'], dtype='object')

In [7]:
df

,epoch,TP,FP,TN,FN,apcer,bpcer,acer,accuracy
0,1,405,353,3958,29,0.066820,0.081884,0.074352,0.919494
1,2,419,747,3564,15,0.034562,0.173278,0.103920,0.839410
2,3,365,2447,1864,69,0.158986,0.567618,0.363302,0.469758
3,5,396,160,4151,38,0.087558,0.037114,0.062336,0.958272
4,6,380,83,4228,54,0.124424,0.019253,0.071839,0.971128
5,7,398,205,4106,36,0.082949,0.047553,0.065251,0.949210
6,8,404,444,3867,30,0.069124,0.102992,0.086058,0.900105
7,9,409,719,3592,25,0.057604,0.166783,0.112193,0.843203
8,10,397,171,4140,37,0.085253,0.039666,0.062460,0.956164
9,11,399,255,4056,35,0.080645,0.059151,0.069898,0.938883


In [6]:
df['batch_directory'].unique()


KeyError: 'batch_directory'

In [ ]:
# Apply classification to batch_directory
df['category'] = df['batch_directory'].apply(lambda x: classify_dataset(x)[0])
df[['batch_directory', 'category']].drop_duplicates()

,batch_directory,category
0,batch_datacollection_20230214_none_recapture,datacollection
4500,batch_production_20240206_none_1000,production


In [ ]:
# All batch directories to classify
batch_dirs = [       #"batch_datacollection_202407_production20240625none_whitebg/batch_datacollection_202407_production20240625none_whitebg.csv",
                     "2024-07-whitebg_fraud/index_annotation_mykadfront_test_set.csv",
                    #"batch_datacollection_202411_inkjet_printer/batch_datacollection_202411_inkjet_printer.csv",
                    "2024-11-inkjet_printer_eval/index_annotation_mykadfront.csv",
                    "batch_datacollection_20230214_none_recapture/batch_datacollection_20230214_recapture_eval_set.csv",
                    "batch_datacollection_20231023_wiseai_recapture_evalfixorigrotatealif/index_annotation_full.csv",
                    "batch_datacollection_20231219_wiseai_recapture_10IC/index_annotation_eval_set.csv",
                    "batch_datacollection_20240303_wiseai_recapture_whitepaperbg/index_annotation_WhitePaper_eval_set.csv",
                    "batch_datacollection_20240303_wiseai_recapture_whitepaperbg/index_annotation_WhitePaper_eval_set_square.csv",
                    "batch_datacollection_20240409_production202208redone_fullpagecolorprinteval/index_annotation_square.csv",
                    "batch_datacollection_20240409_production202208redone_fullpagecolorprinteval/index_annotation_w_genuine.csv",
                    "batch_issue_20230322_none_recapture_colorprint/index_annotation_mykadfront.csv",
                    "batch_issue_20240403_none_recapture_replaymobile/index_annotation_mykadfront.csv",
                    "batch_issue_20240408_partner_recapture_fullcolorprint/index_annotation_mykadfront.csv",
                    "batch_issue_20240412_partner_both_general/index_annotation_mykadfront.csv",
                    "batch_issue_20240704_snt_both_colorghostwhitebg/index_annotation_mykadfront.csv",
                    "batch_issue_20240704_snt_fullpagecolorprint/index_annotation_mykadfront.csv",
                    "batch_issue_20240704_snt_replay/index_annotation_mykadfront.csv",
                    "batch_issue_20240726_bnm_colorprint/batch_issue_20240726_bnm_colorprint.csv", ## not found 
                    "batch_issue_20240823_snt_app/index_annotation_mykadfront.csv",
                    "batch_issue_20241202_snt_mobile/index_annotation_mykadfront.csv",
                    "batch_issue_20241203_postdigi_colorprint/index_annotation_mykadfront.csv",
                    "batch_issue_set/index_annotation_mykadfront_background.csv",
                    "batch_production_202208_redone/index_annotation.csv",
                    "batch_production_20240206_none_1000/index_annotation_mykadfront_annotated_eval_set.csv",
                    "batch_production_20240405_none_merged/index_annotation.csv",
                    "batch_production_20240513_20240519_snt_random/batch_production_20240513_20240519_snt_random.csv", ### not found 
                    #"batch_production_20240625_whitebg_genuine/batch_production_20240625_whitebg_genuine.csv",
                    "production_20240625_whitebg_genuine/index_annotations_mykadfront_filtered_train_set.csv",
                    "batch_production_20240726_bnm_colorprint/batch_production_20240726_bnm_colorprint.csv", ## not found 
                    "batch_production_20250416_none_mypr/index_annotation_mypr_crop_eval.csv",
                    "batch_production_20250416_none_mytentera/index_annotation_mytentera_crop_eval.csv",
                    "batch_production_20250514_none_mytentera/index_annotation_mytentera_crop_eval.csv",
                    #"batch_datacollection_20250530_printed_cutout/batch_datacollection_20250530_printed_cutout.csv",
                    "2025-05-printed_cutout/index_annotation_mykadfront_eval.csv",
                    #"batch_datacollection_20250707_fakeid_v2/batch_datacollection_20250707_fakeid_v2.csv",
                    "2025-07-fakeidtester/index_annotation_mykadfront_eval_v2.csv",
                    #"batch_datacollection_202501_inkjet_printer_cutout_augmentedmaskedbw/batch_datacollection_202501_inkjet_printer_cutout_augmentedmaskedbw.csv",
                    "2025-01-inkjet_printer_cutout_eval/index_annotation_mykadfront_augmented_masked_bw.csv",         
                    "batch_production_20240206_none_1000/index_annotation_mykadfront_annotated_eval_set_augmented_masked_bw.csv",
                    #"grayscale_print_cutout_test_plan_subject/grayscale_print_cutout_test_plan_subject.csv",
                    "1.4 Grayscale Print Cutout Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"colorprint_cutout_test_plan_subject/colorprint_cutout_test_plan_subject.csv",
                    "1.5 Color Print Cutout Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"color_print_test_plan_subject/color_print_test_plan_subject.csv",
                    "1.3 Color Print Test/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"grayscale_print_test_plan_subject/grayscale_print_test_plan_subject.csv",
                    "1.2 Grayscale Print Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"color_print_with_background2_test_plan_subject/color_print_with_background2_test_plan_subject.csv",
                    "1.6.1 Color Print with Background 2 Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"replay_monitor_test_plan_subject/replay_monitor_test_plan_subject.csv",
                    "1.10 Replay Monitor Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"replay_mobile_test_plan_subject/replay_mobile_test_plan_subject.csv",
                    "1.11 Replay Mobile Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"replay_tablet_test_plan_subject/replay_tablet_test_plan_subject.csv",
                    "1.12 Replay Tablet Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    #"genuine_with_background_test_plan_subject/genuine_with_background_test_plan_subject.csv",
                    "1.1 Genuine with Background Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv",
                    "20250804_myekyc_fullday/annotated_filtered_annotated_index_annotation_mykadfront_genuine.csv",                
                    "20250804_myekyc_fullday/annotated_filtered_annotated_index_annotation_mykadfront_print.csv",
                    "20250805_myekyc_fullday/annotated_index_annotation_mykadfront_cutout.csv", ## notfound 
                    "20250805_myekyc_fullday/annotated_index_annotation_mykadfront_genuine.csv", ## not found
                    "20250805_myekyc_fullday/annotated_index_annotation_mykadfront_printed.csv", ## not found 
                    "20250805_myekyc_fullday/annotated_index_annotation_mykadfront_replay.csv", ## not found 
                    "20250806_myekyc_owntester/annotated_index_annotation_mykadfront_v3_genuine.csv",
                    "20250806_myekyc_owntester/annotated_index_annotation_mykadfront_v3_print.csv",
                    "20250806_myekyc_owntester/annotated_index_annotation_mykadfront_v3_replay.csv",
                    "20250808_genuine&cutout/annotated_index_annotation_genuine.csv",
                    "20250808_genuine&cutout/annotated_index_annotation_print_cutout.csv",
                    "20250808_replay_tablet/annotated_index_annotation_genuine.csv",
                    "20250808_replay_tablet/annotated_index_annotation_replay.csv",
                    "20250808_tamper_face/annotated_index_annotation_genuine.csv",
                    "20250807_genuine/annotated_index_annotation.csv",
                    "20250807_tamper_4/annotated_index_annotation.csv",
                    "20250811_genuine_ekyc/filtered_index_annotation.csv",
                    "20250811_Replay_FullPage/filtered_index_annotation_v2_print.csv",
                    "20250811_Replay_FullPage/filtered_index_annotation_v2_replay.csv",
                    "20250811_tamper_face/filtered_index_annotation.csv",
                    "20250813_grayscale/annotated_annotated_index_annotation_mykadfront_print.csv",
                    "20250813_grayscale/annotated_annotated_index_annotation_mykadfront_replay.csv",
                    "20250813_grayscale/annotated_annotated_index_annotation_mykadfront_grayscale.csv",
                    "20250813_wiseai_myid_test-dry_run/index_annotation_mykadfront_v4_print.csv",
                    "20250813_wiseai_myid_test-dry_run/index_annotation_mykadfront_v4_replay.csv",
                    "20250813_wiseai_myid_test-dry_run/index_annotation_mykadfront_v4_genuine.csv",
                    "batch_issue_20250910_bmmb_both_general/index_annotation_mykadfront_v3.csv",
                    "batch_production_20250830_20250903_myid/annotated_index_annotation_mykadfront_v2_genuine.csv",
                    "batch_production_20250830_20250903_myid/annotated_index_annotation_mykadfront_v2_print.csv",
                    "batch_production_20250830_20250903_myid/annotated_index_annotation_mykadfront_v2_replay.csv",
                    ]

# Classify all
results = []
for name in batch_dirs:
    category, weights = classify_dataset(name)
    results.append({
        'batch_directory': name,
        'category': category,
        'apcer_weight': weights['apcer'],
        'bpcer_weight': weights['bpcer'],
    })

result_df = pd.DataFrame(results)
display(result_df)

,batch_directory,category,apcer_weight,bpcer_weight
0,2024-07-whitebg_fraud/index_annotation_mykadfr...,unknown,1.0000,1.0
1,2024-11-inkjet_printer_eval/index_annotation_m...,unknown,1.0000,1.0
2,batch_datacollection_20230214_none_recapture/b...,datacollection,-1.0000,-1.0
3,batch_datacollection_20231023_wiseai_recapture...,datacollection,-1.0000,-1.0
4,batch_datacollection_20231219_wiseai_recapture...,datacollection,-1.0000,-1.0
...,...,...,...,...
68,20250813_wiseai_myid_test-dry_run/index_annota...,unknown,1.0000,1.0
69,batch_issue_20250910_bmmb_both_general/index_a...,issue,-0.0769,0.0
70,batch_production_20250830_20250903_myid/annota...,production,-1.0000,-1.0
71,batch_production_20250830_20250903_myid/annota...,production,-1.0000,-1.0


In [ ]:
result_df[result_df['batch_directory']=='']

,batch_directory,category,apcer_weight,bpcer_weight


In [ ]:
# Show unresolved (unknown category)
unresolved = result_df[result_df['category'] == 'unknown']
print(f"Unresolved: {len(unresolved)} / {len(result_df)}")
unresolved

Unresolved: 32 / 73


,batch_directory,category,apcer_weight,bpcer_weight
0,2024-07-whitebg_fraud/index_annotation_mykadfr...,unknown,1.0,1.0
1,2024-11-inkjet_printer_eval/index_annotation_m...,unknown,1.0,1.0
25,production_20240625_whitebg_genuine/index_anno...,unknown,1.0,1.0
30,2025-05-printed_cutout/index_annotation_mykadf...,unknown,1.0,1.0
31,2025-07-fakeidtester/index_annotation_mykadfro...,unknown,1.0,1.0
32,2025-01-inkjet_printer_cutout_eval/index_annot...,unknown,1.0,1.0
34,1.4 Grayscale Print Cutout Test Plan/cleaned_f...,unknown,1.0,1.0
35,1.5 Color Print Cutout Test Plan/cleaned_filte...,unknown,1.0,1.0
36,1.3 Color Print Test/cleaned_filtered_annotate...,unknown,1.0,1.0
37,1.2 Grayscale Print Test Plan/cleaned_filtered...,unknown,1.0,1.0
